<div style="padding: 20px; background: linear-gradient(90deg, #000000 0%, #434343 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">⚖️ Module 6.5: Reranking (Cross-Encoders)</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Using 100% free, local HuggingFace Cross-Encoders to perfectly sort results.</p>
</div>

---

## 1. Bi-Encoders vs Cross-Encoders

Up until now, we have used **Bi-Encoders** (Standard Embeddings). 
- **Bi-Encoder**: Compares Vector A vs Vector B. Extremely fast. Can search millions of records instantly. But it's somewhat mathematically "dumb" regarding complex context.
- **Cross-Encoder**: Takes the Query AND the Document and feeds them into the neural network *together*. Extremely slow. You cannot run this on a million records.

## 2. The Two-Stage RAG Pipeline
To get the best of both worlds, Production RAG pipelines do this:
1. **Stage 1**: Use Bi-Encoders (Chroma) to quickly fetch the top `k=20` documents out of millions.
2. **Stage 2**: Use a Cross-Encoder (Reranker) to meticulously re-score and sort those 20 documents, passing only the absolute best `k=5` to the LLM.

In [1]:
from sentence_transformers import CrossEncoder
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# 1. Setup Standard Bi-Encoder Database
docs = [
    Document(page_content="Apple just released a new smartphone."),
    Document(page_content="Apples are a delicious red fruit."),
    Document(page_content="Steve Jobs was the CEO of Apple."),
    Document(page_content="I ate an apple pie for dessert."),
]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="rerank_demo")

query = "Who led the company that makes the iPhone?"

print("--- STAGE 1: Standard Search (Bi-Encoder) ---")
# We fetch 4 candidates.
stage_1_results = vs.similarity_search(query, k=4)
for i, d in enumerate(stage_1_results):
    print(f"{i+1}. {d.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- STAGE 1: Standard Search (Bi-Encoder) ---
1. Apple just released a new smartphone.
2. Steve Jobs was the CEO of Apple.
3. Apples are a delicious red fruit.
4. I ate an apple pie for dessert.


## 3. Implementing the Local Cross-Encoder
Instead of paying for Cohere, we will use the `cross-encoder/ms-marco-TinyBERT-L-2-v2` model from HuggingFace for free.

In [2]:
print("Loading Free Local Cross-Encoder...")
# This is a very tiny, fast cross-encoder perfect for CPUs
reranker = CrossEncoder("cross-encoder/ms-marco-TinyBERT-L-2-v2", max_length=512)

# To use a CrossEncoder, we must pass it pairs: [[Query, Doc1], [Query, Doc2], ...]
pairs = [[query, doc.page_content] for doc in stage_1_results]

# Generate scores for the pairs
scores = reranker.predict(pairs)

# Attach the scores back to the documents
scored_docs = list(zip(stage_1_results, scores))

# Sort by score (Highest score = best match)
scored_docs.sort(key=lambda x: x[1], reverse=True)

print("\n--- STAGE 2: Reranked Search (Cross-Encoder) ---")
for doc, score in scored_docs:
    print(f"[Score: {score:+.4f}] {doc.page_content}")

Loading Free Local Cross-Encoder...


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

E:\001_Github_Repo_all\Advanced-RAG-Systems\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mohdf\.cache\huggingface\hub\models--cross-encoder--ms-marco-TinyBERT-L-2-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


--- STAGE 2: Reranked Search (Cross-Encoder) ---
[Score: -9.7922] Steve Jobs was the CEO of Apple.
[Score: -11.1565] Apple just released a new smartphone.
[Score: -11.5174] I ate an apple pie for dessert.
[Score: -11.5631] Apples are a delicious red fruit.
